# Chapter 20 — Multi-GPU with NCCL & ZeRO

> Course: **llm.c — Zero to Hero**, Chapter 20 of ~20.  **Final chapter.**
> Builds on Ch 8 (parameters/grads/optimizer state), Ch 18 (GPU AdamW + global norm), Ch 19 (the training loop), Ch 20a (multi-GPU primitives).

GPT-2 124M fits easily on one GPU. **GPT-3 175B** does not — its FP16 weights alone are 350 GB, and AdamW's master+m+v add another **2.1 TB**. You need many GPUs working together. This chapter is how `llm.c` does that.

The frustrating thing about multi-GPU chapters is they're usually unrunnable on a normal machine — so they stay abstract. We won't do that. This box has **one** GPU, but every idea here is *arithmetic on arrays*, and we'll **simulate the collectives on one machine** — `R` "ranks" as `R` arrays in one program, doing the same math NCCL does across GPUs — and **verify ZeRO-1 produces bit-identical results to plain DDP** while using `1/R` of the optimizer memory.

We'll cover:

1. **DDP (Distributed Data Parallel)** — each GPU holds a full model copy, processes its own batch shard, and gradients are **all-reduced** (averaged) across GPUs.
2. **NCCL** — NVIDIA's GPU-to-GPU collective library (all-reduce, reduce-scatter, all-gather).
3. **ZeRO-1** — shard the optimizer state (master, m, v) across GPUs. Cuts optimizer memory by `N` for `N` GPUs at **zero** change to the result.

### Learning objectives

By the end of this chapter you will:

- Explain DDP and **show** that all-reduce(avg) of per-GPU gradients equals the full-batch gradient.
- Compute the per-GPU memory profile for any model size and GPU count, and see why ZeRO is mandatory at scale.
- **Simulate** the ZeRO-1 sequence (reduce-scatter → sharded AdamW → all-gather) and verify it equals DDP bit-for-bit.
- Read `llmc/zero.cuh::multi_gpu_async_reduce_gradient` and identify the all-reduce vs reduce-scatter paths.


In [ ]:
!mkdir -p course/ch20_build


## 1. Concept — Distributed Data Parallel (DDP)

Eight GPUs, identical models. Each GPU gets a fraction of the global batch:

```
GPU 0:  forward(x[0:B/8])   -> loss -> backward -> grad_0   (mean over its local batch)
GPU 1:  forward(x[B/8:2B/8])-> loss -> backward -> grad_1
...
GPU 7:  forward(x[7B/8:B])  -> loss -> backward -> grad_7

NCCL all-reduce (AVERAGE): every GPU now holds  (grad_0 + ... + grad_7) / 8
Optimizer step: each GPU runs the SAME AdamW on the SAME averaged grad -> identical params
```

Because each GPU computes the **mean** loss over its own `B/8` examples, the average of the eight per-GPU gradients is exactly the gradient you'd get from one batch of `B` on a single giant GPU (Ch 19's accumulation argument, now *across* GPUs). The all-reduce is the only inter-GPU communication.

The catch: each GPU still holds a **full copy of the model + optimizer state**. So DDP gives you `N×` compute and `N×` effective batch — but **no memory savings**. For GPT-2 (≈2 GB of state) that's fine. For GPT-3 (≈2.8 TB) it's hopeless. Hence ZeRO.


## 2. Concept — The NCCL Primitives `llm.c` Uses

NCCL (NVIDIA Collective Communications Library) provides MPI-like collectives, GPU-aware, running over NVLink/PCIe with no host involvement:

| Op | What it does | Used for |
|---|---|---|
| `ncclAllReduce` | Combine a buffer across all ranks; **every** rank gets the result | DDP gradient sync (`ncclAvg`) |
| `ncclReduceScatter` | Combine across all ranks; rank `r` gets only **its slice** of the result | ZeRO-1 gradient sync (`ncclAvg`) |
| `ncclAllGather` | Concatenate each rank's slice; everyone gets the **full** buffer | ZeRO-1 weight broadcast after the step |
| `ncclBroadcast` | Send rank-0's buffer to all others | Init: distribute starting weights |

> **Fidelity note:** `llm.c` reduces with **`ncclAvg`**, not `ncclSum` — the collective averages the gradients for you, matching the "mean over the global batch" semantics directly. (You can confirm this in `llmc/zero.cuh` below.)

The four collectives compose into two recipes: **DDP** = one `AllReduce`; **ZeRO-1** = `ReduceScatter` (before the step) + `AllGather` (after). The next sections build and verify both.


> **New API — the NCCL collective signature.** Every NCCL collective has the same shape:
>
> ```c
> ncclAllReduce(sendbuf, recvbuf, count, datatype, op, comm, stream);
> ```
>
> `sendbuf`/`recvbuf` are **device** pointers; `count` is an **element** count (not bytes); `op` is the reduction (`ncclAvg`, `ncclSum`, …); `comm` is the communicator that ties the ranks together; and `stream` makes the call **asynchronous** — it queues on a CUDA stream and returns immediately, so the all-reduce can overlap the backward pass. `ncclReduceScatter` and `ncclAllGather` take the same arguments; only the data movement differs. `ncclGroupStart()` / `ncclGroupEnd()` batch several collectives (one per tensor) into a single fused network operation.


ZeRO-1 is the recipe worth picturing — reduce-scatter hands each rank only its slice of the averaged gradient, each rank runs AdamW on just that slice, and all-gather stitches the updated weights back together so everyone ends with the full vector:

```mermaid
flowchart LR
  subgraph G["each rank's full local grad (P = 16)"]
    g0["rank0"]
    g1["rank1"]
    g2["rank2"]
    g3["rank3"]
  end
  g0 --> RS["ReduceScatter (avg)"]
  g1 --> RS
  g2 --> RS
  g3 --> RS
  RS --> s0["rank0 owns avg[0:4]"]
  RS --> s1["rank1 owns avg[4:8]"]
  RS --> s2["rank2 owns avg[8:12]"]
  RS --> s3["rank3 owns avg[12:16]"]
  s0 --> a0["AdamW on slice<br/>master/m/v[0:4] only"]
  s1 --> a1["AdamW slice[4:8]"]
  s2 --> a2["AdamW slice[8:12]"]
  s3 --> a3["AdamW slice[12:16]"]
  a0 --> AG["AllGather"]
  a1 --> AG
  a2 --> AG
  a3 --> AG
  AG --> full["every rank: full updated params[0:16]"]
```

Each rank only ever stores `master/m/v` for its own quarter — that is the `1/R` optimizer-memory win, and the math is bit-identical because AdamW is elementwise.


> **▶ Watch first — the whole picture in ~45 seconds.** This animation builds the per-GPU memory bar (params + grads + optimizer state, in the style of **Figure 1** of the [ZeRO paper](https://arxiv.org/abs/1910.02054)), shows why DDP replicates it on every GPU, then shards the optimizer state step by step — **ReduceScatter → sharded AdamW → AllGather** — and finishes by placing ZeRO-1 in the ZeRO family (`Pos`, `Pos+g`, `Pos+g+p`).

<video src="course/videos/ch20_zero1.mp4" controls width="720" poster="course/videos/ch20_zero1_thumbnail.png"></video>

*If the player doesn't load (e.g. on GitHub, which strips `<video>`), open [`course/videos/ch20_zero1.mp4`](course/videos/ch20_zero1.mp4) directly. The animation is generated from [`course/videos/ch20_zero1.py`](course/videos/ch20_zero1.py) with [Manim](https://github.com/3b1b/manim) — example numbers (Ψ=7.5B, K=12, Nd=64 → 120 GB → 31.4 GB per GPU) follow Figure 1.*


## 3. Simulate DDP — All-Reduce == Full Batch

Below, one program plays the role of `R` GPUs. Each "rank" computes the gradient of the **same** linear model over its **own** data shard (the same model as Chapter 19). Then we **average** the per-rank gradients — exactly what `ncclAllReduce(..., ncclAvg)` does across the wire — and check the result equals the gradient computed over the *entire* dataset on one device.


**Concrete example — all-reduce by hand.** Two ranks, gradient vectors of length 3:

- rank0 local grad = `[1, 2, 3]`  (mean over its half of the batch)
- rank1 local grad = `[3, 6, 9]`  (mean over its half)

`AllReduce(avg)` leaves **every** rank holding `([1,2,3] + [3,6,9]) / 2 = [2, 4, 6]`. That is identical to the gradient you'd get by running both halves as one batch on a single giant GPU — which is exactly why each replica then takes the *same* AdamW step and the model copies never drift apart.


In [ ]:
%%writefile course/ch20_build/ddp_sim.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// Mean gradient of a linear model over samples [s0, s1): grad[j] = (1/n) sum (dot(w,x)-t) x_j.
// With s0=0,s1=N this is the FULL-batch gradient; with a shard it's one rank's local gradient.
__global__ void local_grad(float* grad, const float* X, const float* w, const float* t,
                           int D, int s0, int s1) {
    int j = blockIdx.x*blockDim.x + threadIdx.x;
    if (j >= D) return;
    int n = s1 - s0; float g = 0.0f;
    for (int s = s0; s < s1; s++) {
        float p = 0.0f; for (int k=0;k<D;k++) p += w[k]*X[s*D+k];
        g += (p - t[s]) * X[s*D+j];
    }
    grad[j] = g / (float)n;          // MEAN over this rank's local batch
}

int main(void) {
    int N = 4096, D = 8, R = 4, per = N / R;     // N samples, D params, R "GPUs"
    float *X=(float*)malloc(N*D*4), *w=(float*)malloc(D*4), *t=(float*)malloc(N*4);
    for (int s=0;s<N;s++){ for(int k=0;k<D;k++) X[s*D+k]=sinf(0.11f*(s+1)*(k+1)); t[s]=cosf(0.05f*(s+1)); }
    for (int k=0;k<D;k++) w[k]=0.1f*(k-4);
    float *dX,*dw,*dt,*dg; cudaMalloc(&dX,N*D*4);cudaMalloc(&dw,D*4);cudaMalloc(&dt,N*4);cudaMalloc(&dg,D*4);
    cudaMemcpy(dX,X,N*D*4,cudaMemcpyHostToDevice);cudaMemcpy(dw,w,D*4,cudaMemcpyHostToDevice);cudaMemcpy(dt,t,N*4,cudaMemcpyHostToDevice);

    // Reference: full-batch gradient over all N samples (one giant GPU)
    float ref[8]; local_grad<<<1,D>>>(dg, dX,dw,dt, D, 0, N); cudaMemcpy(ref,dg,D*4,cudaMemcpyDeviceToHost);

    // Each rank computes its LOCAL gradient over its shard.
    float grads[4][8];
    for (int r=0;r<R;r++){
        local_grad<<<1,D>>>(dg, dX,dw,dt, D, r*per, r*per+per);
        cudaMemcpy(grads[r], dg, D*4, cudaMemcpyDeviceToHost);
    }
    // NCCL all-reduce with ncclAvg: every rank ends holding the AVERAGE of the R local grads.
    float reduced[8];
    for (int j=0;j<D;j++){ float s=0; for(int r=0;r<R;r++) s+=grads[r][j]; reduced[j]=s/(float)R; }

    float maxdiff=0; for(int j=0;j<D;j++) maxdiff=fmaxf(maxdiff, fabsf(reduced[j]-ref[j]));
    printf("rank0 local grad[0..3] : %+.5f %+.5f %+.5f %+.5f\n", grads[0][0],grads[0][1],grads[0][2],grads[0][3]);
    printf("all-reduced  grad[0..3] : %+.5f %+.5f %+.5f %+.5f   (avg over %d ranks)\n", reduced[0],reduced[1],reduced[2],reduced[3],R);
    printf("full-batch   grad[0..3] : %+.5f %+.5f %+.5f %+.5f   (one giant GPU)\n", ref[0],ref[1],ref[2],ref[3]);
    printf("max |all-reduced - full-batch| = %.2e   -> %s\n", maxdiff, maxdiff<1e-5 ? "PASS" : "FAIL");
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch20_build/ddp_sim course/ch20_build/ddp_sim.cu && ./course/ch20_build/ddp_sim


The per-rank gradient (computed from just 1/4 of the data) is **different** from the full gradient — but **averaging the four** reconstructs the full-batch gradient exactly. That single `ncclAllReduce(..., ncclAvg)` is the entirety of DDP's communication. Every rank now holds the identical averaged gradient, so every rank's subsequent AdamW step produces identical weights — the replicas never drift.


## 4. The Memory Wall — Why DDP Breaks, Runnably

DDP replicates everything. Recall the per-parameter storage during mixed-precision training (Ch 17):

- **working params** (BF16): 2 bytes — needed on every GPU for forward/backward, can't be sharded.
- **gradients** (BF16): 2 bytes — needed on every GPU during backward.
- **optimizer state**: master (FP32) + Adam `m` (FP32) + Adam `v` (FP32) = **12 bytes** — only touched *during the optimizer step*, so it **can** be sharded.

Optimizer state is `3×` the size of the model in FP32 — the dominant term. ZeRO-1 shards exactly that. Plug your own numbers in:


In [ ]:
def per_gpu_gb(num_params: int, num_gpus: int, zero_stage: int) -> dict[str, float]:
    # Per-GPU training memory (GB) for mixed-precision AdamW, with optional ZeRO-1 sharding.
    GB = 1_000_000_000   # decimal GB, to match the canonical "350 GB weights / 2.1 TB optimizer" figures
    working = 2 * num_params                      # BF16 params, replicated on every GPU
    grads   = 2 * num_params                      # BF16 grads, replicated (ZeRO-1 does not shard these)
    shard   = num_gpus if zero_stage == 1 else 1  # ZeRO-1 shards optimizer state across GPUs
    opt     = 3 * 4 * num_params / shard          # master + m + v, each FP32 (4 bytes)
    return {"working": working/GB, "grads": grads/GB, "optimizer": opt/GB,
            "total": (working+grads+opt)/GB}

for name, P in [("GPT-2 124M", 124_000_000), ("GPT-3 175B", 175_000_000_000)]:
    print(f"\n{name}  ({P:,} params)")
    print(f"  {'config':<22}{'working':>9}{'grads':>9}{'optimizer':>11}{'total':>9}  (GB/GPU)")
    for label, gpus, zs in [("DDP, 1 GPU",1,0), ("DDP, 8 GPUs",8,0), ("ZeRO-1, 8 GPUs",8,1), ("ZeRO-1, 64 GPUs",64,1)]:
        m = per_gpu_gb(P, gpus, zs)
        print(f"  {label:<22}{m['working']:>9.3f}{m['grads']:>9.3f}{m['optimizer']:>11.3f}{m['total']:>9.3f}")


Read the GPT-3 rows: under **DDP the optimizer state alone is ~2100 GB (2.1 TB) per GPU** — it cannot fit on anything (an H100 has 80 GB). Note DDP at 8 GPUs is *identical* to 1 GPU — replication buys zero memory relief. **ZeRO-1 across 64 GPUs cuts the optimizer term by 64×** to ~33 GB, which (plus the un-shardable working+grads) finally fits. The working params and gradients stay replicated, so they set the floor — that's why ZeRO-2/3 (which also shard grads, then params) exist for the truly enormous models. `llm.c` implements **ZeRO-1**.


## 5. Simulate ZeRO-1 — Same Answer as DDP, 1/R the Memory

The leap people don't believe until they see it: **ZeRO-1 changes the memory footprint but not a single output bit.** Because AdamW is *elementwise*, updating parameter `j` needs only `grad[j]`, `master[j]`, `m[j]`, `v[j]` — nothing from its neighbors. So we can hand each rank a disjoint slice and the math is unchanged. The per-step recipe:

1. **`ReduceScatter`(avg)** the gradients → rank `r` gets only the averaged gradient for **its slice** `[r·S, (r+1)·S)` (where `S = P/R`).
2. Each rank stores master/m/v for **its slice only** (`S` elements, not `P`) and runs **AdamW on the slice**.
3. **`AllGather`** the updated slices → every rank reconstructs the full BF16 parameter vector.

Below, one program runs both the **DDP baseline** (full optimizer state, full AdamW on every rank) and the **ZeRO-1 path** (sharded), then checks they're bit-identical and prints the per-rank optimizer memory.


In [ ]:
%%writefile course/ch20_build/zero1_sim.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// One AdamW step over n elements (elementwise; identical whether n=P or n=P/R).
void adamw_step(float* w, float* m, float* v, const float* g, int n,
                float lr, float b1, float b2, float eps, float wd, int t) {
    for (int i = 0; i < n; i++) {
        float mn = b1*m[i] + (1-b1)*g[i];
        float vn = b2*v[i] + (1-b2)*g[i]*g[i];
        m[i]=mn; v[i]=vn;
        float mh = mn/(1-powf(b1,t)), vh = vn/(1-powf(b2,t));
        w[i] -= lr * (mh/(sqrtf(vh)+eps) + wd*w[i]);
    }
}

int main(void) {
    int R = 4, P = 16, S = P / R;          // R ranks, P params, S = shard size per rank
    float lr=0.1f, b1=0.9f, b2=0.95f, eps=1e-8f, wd=0.01f; int t=1;

    // Per-rank local gradients (after each rank's backward over its data shard), then the
    // averaged gradient that both DDP all-reduce and ZeRO-1 reduce-scatter are built from.
    float grad[4][16], gavg[16], master0[16];
    for (int r=0;r<R;r++) for(int j=0;j<P;j++) grad[r][j] = 0.2f*sinf(0.7f*(r+1)*(j+1));
    for (int j=0;j<P;j++){ float s=0; for(int r=0;r<R;r++) s+=grad[r][j]; gavg[j]=s/R; }
    for (int j=0;j<P;j++) master0[j] = 0.5f*cosf(0.3f*j);   // shared initial weights

    // ---------- DDP baseline: every rank holds FULL master/m/v (P each), runs full AdamW ----------
    float w_ddp[16], m_ddp[16]={0}, v_ddp[16]={0};
    for (int j=0;j<P;j++) w_ddp[j]=master0[j];
    adamw_step(w_ddp, m_ddp, v_ddp, gavg, P, lr,b1,b2,eps,wd,t);   // ncclAllReduce gave full gavg
    int ddp_opt_floats = 3 * P;                                    // master+m+v, all P

    // ---------- ZeRO-1: each rank owns only its slice of master/m/v (S each) ----------
    float w_zero[16];
    int zero_opt_floats_per_rank = 3 * S;                          // master+m+v, only the shard
    for (int r=0; r<R; r++) {
        // (1) reduce-scatter(avg): rank r receives gavg[r*S : r*S+S] only
        float gslice[16], ws[16], ms[16]={0}, vs[16]={0};
        for (int i=0;i<S;i++){ gslice[i]=gavg[r*S+i]; ws[i]=master0[r*S+i]; }
        // (2) sharded AdamW: update just this rank's S parameters
        adamw_step(ws, ms, vs, gslice, S, lr,b1,b2,eps,wd,t);
        // (3) all-gather: place this rank's updated slice into the full vector everyone ends up with
        for (int i=0;i<S;i++) w_zero[r*S+i] = ws[i];
    }

    float maxdiff=0; for(int j=0;j<P;j++) maxdiff=fmaxf(maxdiff, fabsf(w_ddp[j]-w_zero[j]));
    printf("updated param[0..5] DDP   : "); for(int j=0;j<6;j++) printf("%+.5f ", w_ddp[j]);  printf("\n");
    printf("updated param[0..5] ZeRO-1: "); for(int j=0;j<6;j++) printf("%+.5f ", w_zero[j]); printf("\n");
    printf("max |DDP - ZeRO-1| = %.2e   -> %s\n", maxdiff, maxdiff==0.0f ? "BIT-IDENTICAL" : (maxdiff<1e-6?"PASS":"FAIL"));
    printf("\nper-rank optimizer state:  DDP = %d floats   ZeRO-1 = %d floats   (%dx less, R=%d)\n",
           ddp_opt_floats, zero_opt_floats_per_rank, ddp_opt_floats/zero_opt_floats_per_rank, R);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch20_build/zero1_sim course/ch20_build/zero1_sim.cu && ./course/ch20_build/zero1_sim


**Bit-identical**, at `1/R` the optimizer memory per rank. That's the whole ZeRO-1 bargain: each rank only ever materializes master/m/v for its own shard, runs AdamW on that shard, and the all-gather stitches the updated weights back together. No approximation, no accuracy cost — purely a different *placement* of the same arithmetic.

The communication bill works out fair, too: DDP does one all-reduce (`2× model size` of traffic); ZeRO-1 does reduce-scatter + all-gather (`1× + 1× = 2× model size`). **Same bytes on the wire, `R×` less optimizer memory.** That's why ZeRO-1 is essentially free and on-by-default for large runs.


## 6. The Real Thing — `multi_gpu_async_reduce_gradient`

Now the simulation maps onto real code. From `llmc/zero.cuh` (trimmed to the two branches):

```cpp
template<int N>
void multi_gpu_async_reduce_gradient(floatX* const (&pointers)[N], const size_t (&sizes)[N],
                                     MultiGpuConfig* config, cudaStream_t compute_stream) {
    if (config->num_processes == 1) return;                  // single GPU: nothing to do

    cudaEventRecord(config->compute_nccl_sync, compute_stream);
    cudaStreamWaitEvent(config->nccl_stream, config->compute_nccl_sync);  // wait for backward
    ncclGroupStart();
    for (int i = 0; i < N; ++i) {
        if (config->zero_stage == 0) {                       // ---- DDP ----
            ncclAllReduce(pointers[i], pointers[i], sizes[i],
                          ncclFloatX, ncclAvg, config->nccl_comm, config->nccl_stream);
        } else if (config->zero_stage == 1) {                // ---- ZeRO-1 ----
            size_t shard = sizes[i] / config->num_processes;
            ptrdiff_t off = (ptrdiff_t)shard * config->process_rank;
            ncclReduceScatter(pointers[i], pointers[i] + off, shard,
                              ncclFloatX, ncclAvg, config->nccl_comm, config->nccl_stream);
        }
    }
    ncclGroupEnd();
}
```

The pivot is exactly your Section 3 vs Section 5: `zero_stage == 0` → `ncclAllReduce` (every rank gets the full average — DDP); `zero_stage == 1` → `ncclReduceScatter` writing into `pointers[i] + off` (each rank gets its slice — ZeRO-1). Both reduce with **`ncclAvg`**.

After the step, ZeRO-1 stitches params back together. From `gpt2_update` in `train_gpt2.cu`:

```cpp
ShardInfo shard = multi_gpu_get_shard_offset(tensor.size, multi_gpu_config, 1);
adamw_update(param_ptr + shard.offset, master_ptr, grad_ptr + shard.offset,
             m_ptr, v_ptr, shard.size, ...);                 // AdamW on THIS rank's shard only
if (zero_stage == 1) {
    ncclAllGather(param_ptr + l*tensor.size,                 // my updated slice ->
                  model->params_memory + tensor.offset + l*tensor.size,  // everyone's full buffer
                  shard.size, ncclFloatX, config->nccl_comm, config->nccl_stream);
}
```

And the memory saving is real in the allocation itself — master/m/v are sized to the **shard**, not the whole model:

```cpp
// train_gpt2.cu, optimizer-state allocation
cudaMalloc(&model->m_memory,      shard_num_parameters * sizeof(float));   // not num_parameters!
cudaMalloc(&model->v_memory,      shard_num_parameters * sizeof(float));
cudaMalloc(&model->master_weights, shard_num_parameters * sizeof(float));
// where shard_num_parameters = total_parameters / num_processes  (ZeRO-1)
```

That single substitution — `shard_num_parameters` instead of `num_parameters` — is the entire memory win you measured in Section 5, in production.


## 7. Going to Production — Running Multi-GPU for Real

The simulations above run on this one-GPU box. To run it **for real** across GPUs you need NCCL installed and ≥2 GPUs (or multiple nodes):

```bash
# Build with NCCL + cuDNN
make train_gpt2cu USE_NCCL=1 USE_CUDNN=1

# Launch on 8 GPUs of one node; -zs 1 enables ZeRO-1
mpirun -np 8 ./train_gpt2cu -zs 1

# Multi-node: same binary, rank/master-IP/world-size supplied via env or MPI launcher
```

`mpirun` spawns 8 processes; each calls `multi_gpu_config_init` to join the shared NCCL communicator (via a common `ncclUniqueId`), then runs the **identical training loop from Chapter 19** with its own rank. The only multi-GPU code is the outer orchestration — a few NCCL calls plus a shard offset for AdamW. **Every per-layer kernel — attention, layernorm, matmul — runs completely unchanged.** That's the punchline: the same code that trains on your RTX 4080 trains on a 2048-GPU cluster.


## 8. Going Bigger — Multi-Node, Same Collectives

One node tops out at ~8 GPUs wired together with **NVLink** (hundreds of GB/s, GPU-to-GPU, no CPU). To go past that — the 64-, 2048-, 100k-GPU runs — you connect *machines* over an **InfiniBand** fabric, which is roughly an order of magnitude slower per link than NVLink. So the network is now **two-level**: fast *inside* a node, slow *between* nodes.

The reassuring part: **your collective code does not change.** The `ncclAllReduce` / `ncclReduceScatter` calls from Sections 3–6 are byte-for-byte identical on 8 GPUs and on 8 nodes. NCCL is **topology-aware** — it discovers the NVLink/InfiniBand layout and runs a **hierarchical** reduction automatically:

1. **Reduce within each node** over NVLink → one partial sum per node.
2. **All-reduce those partial sums across nodes** over InfiniBand — the *only* traffic that crosses the slow fabric.
3. **Broadcast the result back down** within each node over NVLink.

So if a node has `G` GPUs, only `1/G` of the data ever touches InfiniBand. (NCCL is also the **only** backend that supports InfiniBand and GPUDirect RDMA — the NIC DMAs straight out of GPU memory, skipping the host.) Let's simulate that hierarchy on this one box and confirm it lands on the same average a flat all-reduce would.


In [ ]:
%%writefile course/ch20_build/multinode_sim.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

// Hierarchical all-reduce over NODES x GPN ranks -- what real NCCL does across machines.
// The ANSWER equals a flat all-reduce(avg); only the PLACEMENT of network traffic changes:
// most of it stays on fast NVLink, and only per-node partial sums cross slow InfiniBand.
int main(void) {
    int NODES = 2, GPN = 4, R = NODES*GPN, P = 8;     // 2 nodes x 4 GPUs/node = 8 ranks
    float grad[8][8];
    for (int r=0;r<R;r++) for(int j=0;j<P;j++) grad[r][j] = 0.1f*sinf(0.3f*(r+1)*(j+1));

    // ---- Reference: FLAT all-reduce(avg) over all R ranks (Section 3, but R=8) ----
    float flat[8];
    for (int j=0;j<P;j++){ float s=0; for(int r=0;r<R;r++) s+=grad[r][j]; flat[j]=s/(float)R; }

    // ---- Hierarchical all-reduce ----
    // (1) intra-node reduce over NVLink: each node sums its GPN local grads -> a partial sum
    float node_sum[2][8];
    for (int n=0;n<NODES;n++) for(int j=0;j<P;j++){
        float s=0; for(int g=0; g<GPN; g++) s += grad[n*GPN+g][j];
        node_sum[n][j] = s;                            // stays on the node (fast NVLink)
    }
    // (2) inter-node all-reduce over InfiniBand: exchange ONLY the NODES partial sums
    float global[8];
    for (int j=0;j<P;j++){ float s=0; for(int n=0;n<NODES;n++) s+=node_sum[n][j]; global[j]=s/(float)R; }
    // (3) intra-node broadcast over NVLink: every GPU on every node now holds `global`

    float maxdiff=0; for(int j=0;j<P;j++) maxdiff=fmaxf(maxdiff, fabsf(global[j]-flat[j]));
    int crossnode = NODES * P;                          // floats that cross InfiniBand
    int all_data  = R * P;                              // floats that would cross if it were flat
    printf("flat all-reduce  [0..3] : %+.5f %+.5f %+.5f %+.5f   (one big fabric)\n", flat[0],flat[1],flat[2],flat[3]);
    printf("hierarchical     [0..3] : %+.5f %+.5f %+.5f %+.5f   (NVLink + InfiniBand)\n", global[0],global[1],global[2],global[3]);
    printf("max |hier - flat| = %.2e   -> %s\n", maxdiff, maxdiff<1e-5 ? "PASS" : "FAIL");
    printf("\ncross-node floats: hierarchical = %d   vs all rank data = %d   (%dx less InfiniBand traffic)\n",
           crossnode, all_data, all_data/crossnode);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch20_build/multinode_sim course/ch20_build/multinode_sim.cu && ./course/ch20_build/multinode_sim


Same averaged gradient as the flat all-reduce — but the cross-node exchange moved only `NODES×P` floats instead of every rank's vector, i.e. **`GPN×` less traffic on the slow link**. That topology-awareness is *why* NCCL, not a hand-rolled reduce, is what you run at scale; the simulation here flattens the two levels, but the arithmetic and the result are exactly what the library produces across real machines.

**Launching it for real** is a change to the *launcher*, not the code. With `llm.c` you point an MPI launcher at a hostfile; the PyTorch-world equivalent is `torchrun` with node flags:

```bash
# llm.c across 2 nodes x 8 GPUs (16 ranks). NCCL auto-selects NVLink intra-node, InfiniBand inter-node.
mpirun -np 16 --hostfile hosts.txt \
       -x NCCL_IB_HCA=mlx5 -x NCCL_SOCKET_IFNAME=eth0 \   # pin the InfiniBand HCAs + bootstrap NIC
       ./train_gpt2cu -zs 1

# PyTorch equivalent: SAME command on every node, only --node_rank differs
torchrun --nnodes=2 --nproc_per_node=8 --node_rank=0 \
         --master_addr=10.0.0.1 --master_port=29500 train.py     # node 1 -> --node_rank=1
```

Each process binds to a **node-local** GPU via `LOCAL_RANK` (0–7), while its **global** `RANK` (0–15) is its identity in the NCCL communicator. The env vars matter: servers have several NICs, and NCCL "picks the first available interface, which might not be the right one" — pin `NCCL_IB_HCA` / `NCCL_SOCKET_IFNAME` or you silently fall back to slow Ethernet. `NCCL_DEBUG=INFO` prints the topology NCCL chose, which is the first thing to check when a multi-node run is mysteriously slow.


> **How the big labs actually get here.** Nobody's first run is on 100k GPUs. The standard ladder is exactly the one this chapter walks: **1 GPU** (is the math right?) → **1 node / 8 GPUs** (do the NVLink collectives match the single-GPU reference — the very PASS check in Section 3?) → **multi-node** (does the InfiniBand path stay bit-comparable?) → **full cluster**. Bugs change character as you climb: gradient-averaging bugs show up at 8 GPUs; loss spikes from a bad data/optimizer-state combination, stragglers, and silent data corruption only appear at hundreds of nodes.
>
> In parallel, teams *don't* tune the recipe on the big model — they fit **scaling laws on small proxy models** (~10M–3B params) to predict the large run, and lay out the parallelism plan with **no GPUs at all** using PyTorch's meta-device / `DTensor` to validate every shard's shapes before paying for hardware. Your Section 3 and 5 PASS checks are miniatures of the lowest rung of that ladder.


## 9. Translation Bridge

| PyTorch | `llm.c` |
|---|---|
| `torch.nn.parallel.DistributedDataParallel(model)` | DDP via `ncclAllReduce(ncclAvg)` after backward |
| `deepspeed` ZeRO stage 1 | `ncclReduceScatter` → sharded AdamW → `ncclAllGather` |
| `torch.distributed.fsdp.FullyShardedDataParallel` | ZeRO-3 (not in `llm.c` — it implements ZeRO-1 only) |
| `accelerate launch ...` / `torchrun` | `mpirun -np N ...` |
| `dist.all_reduce(g, op=ReduceOp.AVG)` | `ncclAllReduce(..., ncclAvg, ...)` |


## 10. Exercise — Implement Reduce-Scatter

A `ReduceScatter` is an `AllReduce` followed by keeping only your slice. Fill in the TODO so that, for each rank `r`, `out[r]` holds the **average** across ranks of the input gradients, but only for that rank's slice `[r*S, r*S+S)`. The check verifies that concatenating all the slices reconstructs the full averaged gradient (which is what the subsequent all-gather relies on).


In [ ]:
%%writefile course/ch20_build/exercise1.cu
#include <stdio.h>
#include <math.h>

int main(void) {
    int R = 4, P = 16, S = P / R;
    // Per-rank gradient buffers (full length P each), as after local backward.
    float grad[4][16];
    for (int r=0;r<R;r++) for(int j=0;j<P;j++) grad[r][j] = sinf(0.5f*(r+1)*(j+1));

    // out[r] will hold rank r's SLICE (S values) of the averaged gradient.
    float out[4][16];

    // TODO: for each rank r and each i in [0, S):
    //   let j = r*S + i  (global index this rank is responsible for)
    //   out[r][i] = average over all ranks rr of grad[rr][j]
    // (write your reduce-scatter here)
    for (int r=0; r<S; r++) {
        for (int i= 0; i<S; i++) {
            int j = r * S + i;
            float avg = 0.0f; for (int rr=0; rr < R; rr++) avg += grad[rr][j];
            out[r][i] = avg / (float)R;
        }
    }

    // verify: concatenating the slices == full all-reduced (averaged) gradient
    int ok = 1;
    for (int r=0;r<R;r++) for (int i=0;i<S;i++) {
        int j = r*S + i;
        float avg = 0; for (int rr=0;rr<R;rr++) avg += grad[rr][j]; avg /= R;
        if (fabsf(out[r][i] - avg) > 1e-6) ok = 0;
    }
    printf("reduce-scatter slice[rank0] = %.5f %.5f %.5f %.5f\n", out[0][0],out[0][1],out[0][2],out[0][3]);
    printf("%s\n", ok ? "PASS" : "FAIL");
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch20_build/exercise1 course/ch20_build/exercise1.cu && ./course/ch20_build/exercise1


### Solution

In [ ]:
%%writefile course/ch20_build/exercise1_sol.cu
#include <stdio.h>
#include <math.h>

int main(void) {
    int R = 4, P = 16, S = P / R;
    float grad[4][16];
    for (int r=0;r<R;r++) for(int j=0;j<P;j++) grad[r][j] = sinf(0.5f*(r+1)*(j+1));
    float out[4][16];

    // reduce-scatter: each rank keeps only its slice of the averaged gradient
    for (int r=0; r<R; r++) {
        for (int i=0; i<S; i++) {
            int j = r*S + i;
            float avg = 0; for (int rr=0; rr<R; rr++) avg += grad[rr][j];
            out[r][i] = avg / (float)R;
        }
    }

    int ok = 1;
    for (int r=0;r<R;r++) for (int i=0;i<S;i++) {
        int j = r*S + i;
        float avg = 0; for (int rr=0;rr<R;rr++) avg += grad[rr][j]; avg /= R;
        if (fabsf(out[r][i] - avg) > 1e-6) ok = 0;
    }
    printf("reduce-scatter slice[rank0] = %.5f %.5f %.5f %.5f\n", out[0][0],out[0][1],out[0][2],out[0][3]);
    printf("%s\n", ok ? "PASS" : "FAIL");
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch20_build/exercise1_sol course/ch20_build/exercise1_sol.cu && ./course/ch20_build/exercise1_sol


## Further Reading

**Source of truth**

- `llmc/zero.cuh` in this repo — `multi_gpu_async_reduce_gradient` (the all-reduce vs reduce-scatter branches) and `multi_gpu_get_shard_offset`.
- [NCCL — Collective Operations](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/usage/collectives.html) — AllReduce / ReduceScatter / AllGather, the exact primitives `llm.c` uses.
- [_ZeRO: Memory Optimizations Toward Training Trillion Parameter Models_](https://arxiv.org/abs/1910.02054) (Rajbhandari et al.) — the optimizer-state sharding behind ZeRO-1, and stages 2/3.

**Going deeper**

- [NCCL documentation — Overview](https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/overview.html) — topology, NVLink vs PCIe, and how AllReduce maps onto hardware (ring vs tree).
- [_How To Scale Your Model_](https://jax-ml.github.io/scaling-book/) — data / tensor / pipeline parallelism beyond ZeRO-1.


## End of Course — Where to From Here?

You've made it. From "what does `int* x` mean?" in Chapter 1 to *simulating ZeRO-1 reduce-scatter into sharded AdamW and proving it bit-identical to DDP* in Chapter 20.

Concrete next steps:

- **Reproduce the GPT-2 124M run.** `./dev/download_starter_pack.sh` then `./train_gpt2cu` (Ch 19). ~2 hours on a single 4080; watch loss go down.
- **Run `./test_gpt2cu`.** Confirm every layer matches PyTorch byte-for-byte.
- **Fork one kernel.** Replace `layernorm_forward_kernel6` with RMSNorm; verify against PyTorch via `test_gpt2cu`. Best way to consolidate.
- **Read `train_llama3.py`.** Same concepts, different details (RoPE, RMSNorm, SiLU). After GPT-2 it's a small step.
- **Read CUTLASS / CuTe.** Once you grasp `llm.c`'s hand-rolled kernels, CUTLASS shows the next abstraction layer.

### Course summary

| Part | Chapters | What you can do now |
|---|---|---|
| I — C foundations | 1–8 | Read every line of `train_gpt2.c`, derive every backward pass, write CPU AdamW |
| II — CUDA fundamentals | 9–16 | Write CUDA kernels: grid-stride, coalesced loads, warp/block reductions, cuBLAS, Flash Attention |
| III — Production GPU | 17–20 | Read every line of `train_gpt2.cu`, run the full mixed-precision loop, simulate multi-GPU ZeRO-1 |

Thanks for following along. Now go train something.

— *llm.c — Zero to Hero* (Chapters 1–20 complete).
